# Check `dev` stack

Verifies all developer tools are properly installed and functional.
Each section performs a version check followed by a self-contained functional test.

## Version checks

In [ ]:
! quarto --version

In [ ]:
! decktape --version

In [ ]:
! typst --version

In [ ]:
! tippecanoe --version

In [ ]:
! jekyll --version

In [ ]:
! gpq --version

In [ ]:
! jupyter-book --version

## Functional tests

### `quarto` — HTML render

Renders a minimal document to HTML. No network access required.

In [ ]:
import os

qmd = """\
---
title: "GDS Test"
format: html
---

# Test

Quarto functional test.
"""

with open('/tmp/gds_test.qmd', 'w') as f:
    f.write(qmd)

In [ ]:
! quarto render /tmp/gds_test.qmd 2>&1

In [ ]:
assert os.path.exists('/tmp/gds_test.html'), 'quarto render failed: no HTML output'
html_size = os.path.getsize('/tmp/gds_test.html')
assert html_size > 1000, f'quarto render failed: HTML suspiciously small ({html_size} bytes)'
print(f'PASS: quarto rendered HTML ({html_size:,} bytes)')
os.remove('/tmp/gds_test.html')
os.remove('/tmp/gds_test.qmd')

### `decktape` — PDF from reveal.js presentation

Generates a self-contained reveal.js presentation with quarto (no CDN required since
quarto bundles reveal.js locally), then converts it to PDF with decktape.
Uses `--disable-dev-shm-usage` to prevent Chrome crashes in Docker containers
where /dev/shm is limited.

In [ ]:
slides_qmd = """\
---
title: "GDS Slides Test"
format:
  revealjs:
    embed-resources: true
---

## Slide 1

Testing decktape integration.

## Slide 2

Decktape is properly installed.
"""

with open('/tmp/gds_slides.qmd', 'w') as f:
    f.write(slides_qmd)

In [ ]:
! quarto render /tmp/gds_slides.qmd 2>&1

In [ ]:
import subprocess

result = subprocess.run(
    [
        'decktape', 'reveal',
        '--chrome-arg=--no-sandbox',
        '--chrome-arg=--disable-dev-shm-usage',
        '-s', '1280x960',
        'file:///tmp/gds_slides.html',
        '/tmp/gds_slides.pdf',
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('--- stderr ---')
    print(result.stderr)
    raise RuntimeError(f'decktape exited with code {result.returncode}')

In [ ]:
assert os.path.exists('/tmp/gds_slides.pdf'), 'decktape failed: PDF not created'
pdf_size = os.path.getsize('/tmp/gds_slides.pdf')
assert pdf_size > 1000, f'decktape failed: PDF suspiciously small ({pdf_size} bytes)'
print(f'PASS: decktape created PDF ({pdf_size:,} bytes)')
for path in ['/tmp/gds_slides.qmd', '/tmp/gds_slides.html', '/tmp/gds_slides.pdf']:
    if os.path.exists(path):
        os.remove(path)

### `typst` — compile a minimal document to PDF

Compiles an inline Typst document inside a temporary directory, verifies the
generated PDF, and cleans up automatically.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

typst_doc = """\
#set page(width: 120mm, height: 80mm, margin: 10mm)
= GDS Typst Test

This PDF confirms Typst is installed and can compile a minimal document.
"""

with tempfile.TemporaryDirectory(prefix='gds-typst-') as tmpdir:
    tmpdir_path = Path(tmpdir)
    source_path = tmpdir_path / 'test.typ'
    pdf_path = tmpdir_path / 'test.pdf'
    source_path.write_text(typst_doc)

    result = subprocess.run(
        ['typst', 'compile', str(source_path), str(pdf_path)],
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print('--- stderr ---')
            print(result.stderr)
        raise RuntimeError(f'typst exited with code {result.returncode}')

    assert pdf_path.exists(), 'typst failed: PDF not created'
    pdf_bytes = pdf_path.read_bytes()
    assert pdf_bytes.startswith(b'%PDF-'), 'typst failed: output is not a PDF'
    pdf_size = len(pdf_bytes)
    assert pdf_size > 500, f'typst failed: PDF suspiciously small ({pdf_size} bytes)'
    print(f'PASS: typst created PDF ({pdf_size:,} bytes)')

### `tippecanoe` — GeoJSON to MBTiles

Converts an inline GeoJSON (no download required) to a vector tile archive.

In [ ]:
import json

geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [-0.1278, 51.5074]},
            'properties': {'name': 'London'},
        },
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [2.3522, 48.8566]},
            'properties': {'name': 'Paris'},
        },
    ],
}

with open('/tmp/test_points.geojson', 'w') as f:
    json.dump(geojson, f)

In [ ]:
! tippecanoe -o /tmp/test.mbtiles /tmp/test_points.geojson 2>&1

In [ ]:
assert os.path.exists('/tmp/test.mbtiles'), 'tippecanoe failed: .mbtiles not created'
mbtiles_size = os.path.getsize('/tmp/test.mbtiles')
assert mbtiles_size > 0, 'tippecanoe failed: .mbtiles is empty'
print(f'PASS: tippecanoe created MBTiles ({mbtiles_size:,} bytes)')
os.remove('/tmp/test.mbtiles')
os.remove('/tmp/test_points.geojson')

### `jupyter-book` — PDF export (LaTeX and Typst)

Builds a minimal MyST document to PDF two ways: `--pdf` (via LaTeX — `latexmk` +
`xelatex`) and `--typst` (via the `typst` binary). This exercises the full
PDF-export toolchain that Jupyter Book 2.x uses. Unlike the other checks this one
needs network access: MyST fetches its export templates from `api.mystmd.org`.


In [ ]:
import subprocess
import tempfile
from pathlib import Path

# jupyter-book 2.x renders PDF from MyST: --pdf goes through LaTeX (latexmk +
# xelatex), --typst through the typst binary. Both fetch a template over the
# network (all image builds have network anyway).
book_md = """\
---
title: GDS PDF Test
authors:
  - name: GDS
---
# Intro

jupyter-book PDF export functional test. Inline math: $e^{i\\pi} + 1 = 0$.
"""

def _check_pdf(pdfs, label):
    assert pdfs, f'{label}: no PDF produced'
    data = pdfs[0].read_bytes()
    assert data.startswith(b'%PDF-'), f'{label}: output is not a PDF ({pdfs[0]})'
    assert len(data) > 1000, f'{label}: PDF suspiciously small ({len(data)} bytes)'
    return pdfs[0], len(data)

with tempfile.TemporaryDirectory(prefix='gds-jb-') as tmpdir:
    src = Path(tmpdir) / 'gds_jb.md'
    src.write_text(book_md)

    for builder in ('--pdf', '--typst'):
        result = subprocess.run(
            ['jupyter-book', 'build', str(src), builder],
            capture_output=True, text=True, cwd=tmpdir,
        )
        if result.returncode != 0:
            print(result.stdout[-2000:])
            print('--- stderr ---')
            print(result.stderr[-2000:])
            raise RuntimeError(f'jupyter-book build {builder} exited {result.returncode}')

    exports = Path(tmpdir) / '_build' / 'exports'
    latex_pdf, latex_size = _check_pdf(list(exports.glob('*.pdf')), 'jupyter-book --pdf')
    typst_pdf, typst_size = _check_pdf(list(exports.glob('*_typst/*.pdf')), 'jupyter-book --typst')
    print(f'PASS: jupyter-book --pdf   -> {latex_pdf.name} ({latex_size:,} bytes)')
    print(f'PASS: jupyter-book --typst -> {typst_pdf.name} ({typst_size:,} bytes)')
